<a href="https://colab.research.google.com/github/uncannyeng01/yantaramalisonarprojesi_yolo11/blob/main/03_bm3d_sigma_parametreleri.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install bm3d

In [ ]:
import os, json, time, shutil, re, cv2, numpy as np
from PIL import Image
import bm3d
from bm3d import BM3DProfile, BM3DStages

BASE = '/content/drive/MyDrive/sonar_proje'
SRC  = f'{BASE}/dataset_orig'
SIGMALAR = [10]            # bu oturumda: [10] veya [30]
UZANTI = {'.png', '.jpg', '.jpeg', '.bmp'}

dislanan = set()
yol = f'{BASE}/excluded_images.json'
if os.path.exists(yol):
    d = json.load(open(yol))
    parcalar = d if isinstance(d, list) else [x for v in (d.values() if isinstance(d, dict) else [])
                                              for x in (v if isinstance(v, list) else [v])]
    for x in parcalar:
        ad = os.path.basename(str(x))
        dislanan |= {ad, os.path.splitext(ad)[0]}

def read_bgr(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None or img.ndim != 3:
        img = cv2.cvtColor(np.array(Image.open(path).convert('RGB')), cv2.COLOR_RGB2BGR)
    return img

def bm3d_hizli(img_bgr, sigma):
    ycc = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    y = ycc[:, :, 0].astype(np.float32) / 255.0
    prof = BM3DProfile()
    prof.step_ht = 6
    prof.search_window_ht = 19
    yd = bm3d.bm3d(y, sigma_psd=sigma / 255.0, profile=prof,
                   stage_arg=BM3DStages.HARD_THRESHOLDING)
    ycc[:, :, 0] = np.clip(yd * 255.0 + 0.5, 0, 255).astype(np.uint8)
    return cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR)

for sg in SIGMALAR:
    DST = f'{BASE}/dataset_bm3d_s{sg}'
    json.dump(dict(sigma=sg, kanal='Y (YCrCb)', asama='HARD_THRESHOLDING',
                   step_ht=6, search_window_ht=19),
              open(f'{BASE}/bm3d_params_s{sg}.json', 'w'), indent=2)
    hatalar, islenen, t0 = [], 0, time.time()
    isler = []
    for split in ['train', 'val', 'test']:
        os.makedirs(f'{DST}/images/{split}', exist_ok=True)
        os.makedirs(f'{DST}/labels/{split}', exist_ok=True)
        for f in sorted(os.listdir(f'{SRC}/images/{split}')):
            ad, uz = os.path.splitext(f)
            if uz.lower() in UZANTI and f not in dislanan and ad not in dislanan:
                isler.append((split, f, ad))
    print(f'\n=== BM3D σ={sg} | toplam {len(isler)} görüntü ===')
    for n, (split, f, ad) in enumerate(isler, 1):
        cikis = f'{DST}/images/{split}/{ad}.png'
        lbl = f'{SRC}/labels/{split}/{ad}.txt'
        if os.path.exists(lbl):
            shutil.copy(lbl, f'{DST}/labels/{split}/{ad}.txt')
        if os.path.exists(cikis):
            continue
        try:
            out = bm3d_hizli(read_bgr(f'{SRC}/images/{split}/{f}'), sg)
            gecici = cikis + '.tmp.png'
            cv2.imwrite(gecici, out)
            os.replace(gecici, cikis)
            islenen += 1
        except Exception as e:
            hatalar.append((f, str(e)[:80]))
        if n % 10 == 0:
            gecen = time.time() - t0
            print(f'{n}/{len(isler)} | geçen {gecen/60:.1f} dk | hata {len(hatalar)}')
    ornek = f'{BASE}/dataset_bm3d/data.yaml'
    if os.path.exists(ornek):
        txt = re.sub(r'^path:.*$', f'path: {DST}', open(ornek).read(), flags=re.M)
    else:
        txt = f'path: {DST}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n'
    open(f'{DST}/data.yaml', 'w').write(txt)
    json.dump(hatalar, open(f'{DST}/hatalar.json', 'w'), indent=2)
    print(f'σ={sg} bitti | işlenen {islenen} | hata {len(hatalar)} | süre {(time.time()-t0)/3600:.2f} saat')

In [ ]:
import glob
BASE = '/content/drive/MyDrive/sonar_proje'
for s in ['train', 'val', 'test']:
    d = glob.glob(f'{BASE}/dataset_bm3d_s10/images/{s}/*.png')
    t = [f for f in d if f.endswith('.tmp.png')]
    print(s, len(d) - len(t), 'tmp:', len(t))

In [ ]:
import os, json, time, shutil, re, cv2, numpy as np
from PIL import Image
import bm3d
from bm3d import BM3DProfile, BM3DStages

BASE = '/content/drive/MyDrive/sonar_proje'
SRC  = f'{BASE}/dataset_orig'
SIGMALAR = [30]
UZANTI = {'.png', '.jpg', '.jpeg', '.bmp'}

dislanan = set()
yol = f'{BASE}/excluded_images.json'
if os.path.exists(yol):
    d = json.load(open(yol))
    parcalar = d if isinstance(d, list) else [x for v in (d.values() if isinstance(d, dict) else [])
                                              for x in (v if isinstance(v, list) else [v])]
    for x in parcalar:
        ad = os.path.basename(str(x))
        dislanan |= {ad, os.path.splitext(ad)[0]}

def read_bgr(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None or img.ndim != 3:
        img = cv2.cvtColor(np.array(Image.open(path).convert('RGB')), cv2.COLOR_RGB2BGR)
    return img

def bm3d_hizli(img_bgr, sigma):
    ycc = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    y = ycc[:, :, 0].astype(np.float32) / 255.0
    prof = BM3DProfile()
    prof.step_ht = 6
    prof.search_window_ht = 19
    yd = bm3d.bm3d(y, sigma_psd=sigma / 255.0, profile=prof,
                   stage_arg=BM3DStages.HARD_THRESHOLDING)
    ycc[:, :, 0] = np.clip(yd * 255.0 + 0.5, 0, 255).astype(np.uint8)
    return cv2.cvtColor(ycc, cv2.COLOR_YCrCb2BGR)

for sg in SIGMALAR:
    DST = f'{BASE}/dataset_bm3d_s{sg}'
    json.dump(dict(sigma=sg, kanal='Y (YCrCb)', asama='HARD_THRESHOLDING',
                   step_ht=6, search_window_ht=19),
              open(f'{BASE}/bm3d_params_s{sg}.json', 'w'), indent=2)
    hatalar, islenen, t0 = [], 0, time.time()
    isler = []
    for split in ['train', 'val', 'test']:
        os.makedirs(f'{DST}/images/{split}', exist_ok=True)
        os.makedirs(f'{DST}/labels/{split}', exist_ok=True)
        for f in sorted(os.listdir(f'{SRC}/images/{split}')):
            ad, uz = os.path.splitext(f)
            if uz.lower() in UZANTI and f not in dislanan and ad not in dislanan:
                isler.append((split, f, ad))
    for n, (split, f, ad) in enumerate(isler, 1):
        cikis = f'{DST}/images/{split}/{ad}.png'
        lbl = f'{SRC}/labels/{split}/{ad}.txt'
        if os.path.exists(lbl):
            shutil.copy(lbl, f'{DST}/labels/{split}/{ad}.txt')
        if os.path.exists(cikis):
            continue
        try:
            out = bm3d_hizli(read_bgr(f'{SRC}/images/{split}/{f}'), sg)
            gecici = cikis + '.tmp.png'
            cv2.imwrite(gecici, out)
            os.replace(gecici, cikis)
            islenen += 1
        except Exception as e:
            hatalar.append((f, str(e)[:80]))
        if n % 10 == 0:
            print(f'{n}/{len(isler)} | geçen {(time.time()-t0)/60:.1f} dk | hata {len(hatalar)}')
    ornek = f'{BASE}/dataset_bm3d/data.yaml'
    if os.path.exists(ornek):
        txt = re.sub(r'^path:.*$', f'path: {DST}', open(ornek).read(), flags=re.M)
    else:
        txt = f'path: {DST}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: Human\n'
    open(f'{DST}/data.yaml', 'w').write(txt)
    json.dump(hatalar, open(f'{DST}/hatalar.json', 'w'), indent=2)
    print(f'σ={sg} bitti | işlenen {islenen} | hata {len(hatalar)} | süre {(time.time()-t0)/3600:.2f} saat')

In [ ]:
import glob
BASE = '/content/drive/MyDrive/sonar_proje'
for s in ['train', 'val', 'test']:
    d = glob.glob(f'{BASE}/dataset_bm3d_s30/images/{s}/*.png')
    t = [f for f in d if f.endswith('.tmp.png')]
    print(s, len(d) - len(t), 'tmp:', len(t))